# 3.7 — JSON and the `VARIANT` Type

**Chapter 3, section 3.11** (*Working with Semi-Structured Data*), and Exercise 10.

**The question this notebook answers:** application logs, event streams and API responses do
not arrive as flat tables. They arrive as JSON, with nested objects, repeated arrays, and fields
that are present in some records and absent from others. Section 3.11 puts the two answers to
that at opposite ends of a spectrum — a schema declared fully in advance, or the Spark 4.0
`VARIANT` type that defers all knowledge of the schema until the data is read. What does each
actually do to an awkward feed, and what does each cost?

The feed used here is deliberately awkward in the three ways real feeds are:

* most records share a common core (`user`, `items`, `ts`), and a minority carry extra fields
  nobody agreed on in advance;
* one field is a number in most records and a string in one, which is what breaks inference;
* one record is missing a field entirely.

All data is generated in the notebook. Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import json, os, tempfile
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               LongType, ArrayType, DoubleType)

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
WORK = os.path.join(SCRATCH, "ch03-json")
os.makedirs(WORK, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-3.7")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")   # keep printed output clean
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


## The feed

One JSON object per line — the format `spark.read.json` expects, and the format almost every
log shipper produces. Written out to a file so that the reads below are real file reads.

In [2]:
EVENTS = [
    {"event_id": 1, "ts": "2025-06-01T09:15:00Z",
     "user": {"id": 101, "city": "Boston", "age": 31},
     "items": [{"sku": "A-1", "qty": 2, "price": 9.99},
               {"sku": "B-7", "qty": 1, "price": 24.50}]},

    {"event_id": 2, "ts": "2025-06-01T09:17:31Z",
     "user": {"id": 102, "city": "Lisbon", "age": 45},
     "items": [{"sku": "C-3", "qty": 5, "price": 4.00}],
     # a field nobody agreed on in advance
     "experiment": {"bucket": "B", "variant": 3}},

    {"event_id": 3, "ts": "2025-06-01T09:22:04Z",
     "user": {"id": 103, "city": "Boston"},            # no age at all
     "items": []},

    {"event_id": 4, "ts": "2025-06-01T09:31:57Z",
     "user": {"id": 104, "city": "Nairobi", "age": "unknown"},   # a string where a number was
     "items": [{"sku": "A-1", "qty": 1, "price": 9.99}]},

    {"event_id": 5, "ts": "2025-06-01T09:40:12Z",
     "user": {"id": 105, "city": "Lisbon", "age": 28},
     "items": [{"sku": "B-7", "qty": 3, "price": 24.50}],
     "device": {"os": "iOS", "version": "18.2", "push_enabled": True}},
]

EVENTS_PATH = os.path.join(WORK, "events.json")
with open(EVENTS_PATH, "w") as fh:
    for e in EVENTS:
        fh.write(json.dumps(e) + "\n")

print(open(EVENTS_PATH).read())

{"event_id": 1, "ts": "2025-06-01T09:15:00Z", "user": {"id": 101, "city": "Boston", "age": 31}, "items": [{"sku": "A-1", "qty": 2, "price": 9.99}, {"sku": "B-7", "qty": 1, "price": 24.5}]}
{"event_id": 2, "ts": "2025-06-01T09:17:31Z", "user": {"id": 102, "city": "Lisbon", "age": 45}, "items": [{"sku": "C-3", "qty": 5, "price": 4.0}], "experiment": {"bucket": "B", "variant": 3}}
{"event_id": 3, "ts": "2025-06-01T09:22:04Z", "user": {"id": 103, "city": "Boston"}, "items": []}
{"event_id": 4, "ts": "2025-06-01T09:31:57Z", "user": {"id": 104, "city": "Nairobi", "age": "unknown"}, "items": [{"sku": "A-1", "qty": 1, "price": 9.99}]}
{"event_id": 5, "ts": "2025-06-01T09:40:12Z", "user": {"id": 105, "city": "Lisbon", "age": 28}, "items": [{"sku": "B-7", "qty": 3, "price": 24.5}], "device": {"os": "iOS", "version": "18.2", "push_enabled": true}}



---

## 1. Inference: what Spark guesses

`spark.read.json` with no schema samples the file and infers one. It is convenient, and on an
irregular feed it is also where the trouble starts.

In [3]:
inferred = spark.read.json(EVENTS_PATH)
inferred.printSchema()

root
 |-- device: struct (nullable = true)
 |    |-- os: string (nullable = true)
 |    |-- push_enabled: boolean (nullable = true)
 |    |-- version: string (nullable = true)
 |-- event_id: long (nullable = true)
 |-- experiment: struct (nullable = true)
 |    |-- bucket: string (nullable = true)
 |    |-- variant: long (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |-- ts: string (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- age: string (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- id: long (nullable = true)



Two things to notice, and both are the reason section 3.6 prefers a declared schema.

**The union of every field.** `experiment` and `device` appear in the schema even though only
one record carries each, and they are null everywhere else. Spark has widened the schema to the
union of everything it saw — which means the schema of your table is now a function of which
records happened to be in the file this morning.

**`user.age` is a string.** Three records have a number there, one has no `age` at all, and one
has the string `"unknown"` — so the only type that accommodates them all is `STRING`. Every arithmetic use of `age` downstream now needs
a cast, and the cast will be silently wrong on the record that made it a string in the first
place. This is the same failure as the nine-digit postal code in section 3.2, arriving from the
other direction.

In [4]:
inferred.select("event_id", "user.id", "user.city", "user.age").orderBy("event_id").show()

+--------+---+-------+-------+
|event_id| id|   city|    age|
+--------+---+-------+-------+
|       1|101| Boston|     31|
|       2|102| Lisbon|     45|
|       3|103| Boston|   NULL|
|       4|104|Nairobi|unknown|
|       5|105| Lisbon|     28|
+--------+---+-------+-------+



---

## 2. A declared nested schema: schema-on-write

When the structure *is* known, describe it. Objects become `StructType`, repeated fields become
`ArrayType`, and Spark parses each record into typed, nested columns accordingly. Anything that
does not fit the declaration becomes null rather than silently widening the type of the column.

In [5]:
event_schema = StructType([
    StructField("event_id", LongType(), True),
    StructField("ts",       StringType(), True),
    StructField("user", StructType([
        StructField("id",   LongType(),   True),
        StructField("city", StringType(), True),
        StructField("age",  IntegerType(), True),      # declared a number, deliberately
    ]), True),
    StructField("items", ArrayType(StructType([
        StructField("sku",   StringType(), True),
        StructField("qty",   IntegerType(), True),
        StructField("price", DoubleType(),  True),
    ])), True),
])

events = spark.read.schema(event_schema).json(EVENTS_PATH)
events.printSchema()
events.orderBy("event_id").show(truncate=False)

root
 |-- event_id: long (nullable = true)
 |-- ts: string (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- age: integer (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- sku: string (nullable = true)
 |    |    |-- qty: integer (nullable = true)
 |    |    |-- price: double (nullable = true)



+--------+--------------------+--------------------+--------------------------------+
|event_id|ts                  |user                |items                           |
+--------+--------------------+--------------------+--------------------------------+
|1       |2025-06-01T09:15:00Z|{101, Boston, 31}   |[{A-1, 2, 9.99}, {B-7, 1, 24.5}]|
|2       |2025-06-01T09:17:31Z|{102, Lisbon, 45}   |[{C-3, 5, 4.0}]                 |
|3       |2025-06-01T09:22:04Z|{103, Boston, NULL} |[]                              |
|4       |2025-06-01T09:31:57Z|{104, Nairobi, NULL}|[{A-1, 1, 9.99}]                |
|5       |2025-06-01T09:40:12Z|{105, Lisbon, 28}   |[{B-7, 3, 24.5}]                |
+--------+--------------------+--------------------+--------------------------------+



The declared schema did three things at once. It **read the file in one pass** rather than two.
It **ignored `experiment` and `device`** entirely — they are not in the contract, so they do not
appear. And where `age` was `"unknown"` it produced a **null**, because `unknown` is not an
integer, rather than degrading the whole column to text.

That last is the bargain of schema-on-write in one row: the malformed value is caught at the
moment of reading, and the rest of the column keeps its type.

A nested field is reached by dotted notation, and an array field is expanded into one row per
element with `explode` — the structured counterpart of the `flatMap` met earlier.

In [6]:
events.select("event_id", "user.id", "user.city", "user.age").orderBy("event_id").show()

+--------+---+-------+----+
|event_id| id|   city| age|
+--------+---+-------+----+
|       1|101| Boston|  31|
|       2|102| Lisbon|  45|
|       3|103| Boston|NULL|
|       4|104|Nairobi|NULL|
|       5|105| Lisbon|  28|
+--------+---+-------+----+



In [7]:
# explode: one row per array element.  Note that event 3, whose `items` is empty, disappears
# -- explode drops rows with nothing to expand.  `explode_outer` keeps them with a null.
lines = events.select("event_id", F.explode("items").alias("item"))
lines.select("event_id", "item.sku", "item.qty", "item.price").orderBy("event_id", "item.sku").show()

print("revenue per SKU")
(lines.groupBy("item.sku")
      .agg(F.round(F.sum(F.col("item.qty") * F.col("item.price")), 2).alias("revenue"))
      .orderBy(F.desc("revenue")).show())

+--------+---+---+-----+
|event_id|sku|qty|price|
+--------+---+---+-----+
|       1|A-1|  2| 9.99|
|       1|B-7|  1| 24.5|
|       2|C-3|  5|  4.0|
|       4|A-1|  1| 9.99|
|       5|B-7|  3| 24.5|
+--------+---+---+-----+

revenue per SKU


+---+-------+
|sku|revenue|
+---+-------+
|B-7|   98.0|
|A-1|  29.97|
|C-3|   20.0|
+---+-------+



### When only a few fields are wanted

Two older functions serve the case where the JSON is otherwise left as text. `from_json` parses
a string column against a supplied schema; `get_json_object` extracts a single path with no
schema at all. Both have served for years, and both share a limitation: they require the
programmer either to know the schema in advance, or to re-parse the raw text on every access.

In [8]:
raw = spark.read.text(EVENTS_PATH).withColumnRenamed("value", "json_text")

user_schema = StructType([StructField("city", StringType(), True),
                          StructField("id",   LongType(),   True)])

(raw.select(
    F.get_json_object("json_text", "$.event_id").alias("event_id"),
    F.get_json_object("json_text", "$.user.city").alias("city_by_path"),
    F.from_json(F.get_json_object("json_text", "$.user"), user_schema).alias("user_parsed"))
  .orderBy("event_id").show(truncate=False))

+--------+------------+--------------+
|event_id|city_by_path|user_parsed   |
+--------+------------+--------------+
|1       |Boston      |{Boston, 101} |
|2       |Lisbon      |{Lisbon, 102} |
|3       |Boston      |{Boston, 103} |
|4       |Nairobi     |{Nairobi, 104}|
|5       |Lisbon      |{Lisbon, 105} |
+--------+------------+--------------+



---

## 3. `VARIANT`: schema-on-read

Spark 4.0 introduced a type for data whose schema is partial, irregular or evolving. A `VARIANT`
column holds an entire semi-structured value — an arbitrary JSON document, nesting preserved —
in a single column, but it does not hold it as text. It uses a binary encoding that records
where each field is located, so a nested field can be read directly without re-parsing the
surrounding document each time.

`parse_json` builds one from a JSON string.

In [9]:
docs = raw.withColumn("v", F.parse_json(F.col("json_text")))
docs.printSchema()                       # note the column type: `variant`

root
 |-- json_text: string (nullable = true)
 |-- v: variant (nullable = true)



Fields are read from the parsed value with `variant_get`, which takes a path into the document
and the type the extracted value is to be cast to. Omitting the type — which SQL permits —
returns another `VARIANT` rather than a concrete value.

In [10]:
docs.select(
    F.variant_get("v", "$.event_id", "INT").alias("event_id"),
    F.variant_get("v", "$.user.city", "STRING").alias("city"),
    F.try_variant_get("v", "$.user.age", "INT").alias("age"),
    # fields only some records carry: null where absent, no schema change required
    F.try_variant_get("v", "$.experiment.bucket", "STRING").alias("bucket"),
    F.try_variant_get("v", "$.device.os", "STRING").alias("os"),
).orderBy("event_id").show()

+--------+-------+----+------+----+
|event_id|   city| age|bucket|  os|
+--------+-------+----+------+----+
|       1| Boston|  31|  NULL|NULL|
|       2| Lisbon|  45|     B|NULL|
|       3| Boston|NULL|  NULL|NULL|
|       4|Nairobi|NULL|  NULL|NULL|
|       5| Lisbon|  28|  NULL| iOS|
+--------+-------+----+------+----+



That is the payoff. `experiment.bucket` and `device.os` were read straight out of the documents
that have them, with nulls for the documents that do not, and **nothing had to be declared in
advance**. A new field appearing in tomorrow's feed needs no schema migration; it is simply
there to be asked for.

### `variant_get` raises, `try_variant_get` returns null

The pair is the same distinction as `cast` versus `try_cast`, and the choice between them is a
choice about whether a malformed value should stop the job or pass through as a null.

In [11]:
# Executor-side failures are logged at ERROR even when the exception is caught in the
# driver.  The failure below is deliberate, so the log threshold is lifted over it.
spark.sparkContext.setLogLevel("FATAL")
try:
    docs.select(F.variant_get("v", "$.user.age", "INT").alias("age")).show()
except Exception as e:
    print(type(e).__name__)
    print(str(e).split("\n")[0])
finally:
    spark.sparkContext.setLogLevel("ERROR")

print()
print("try_variant_get, on the same column:")
docs.select(F.try_variant_get("v", "$.user.age", "INT").alias("age")).show()

SparkRuntimeException
[INVALID_VARIANT_CAST] The variant value `"unknown"` cannot be cast into `"INT"`. Please use `try_variant_get` instead. SQLSTATE: 22023

try_variant_get, on the same column:
+----+
| age|
+----+
|  31|
|  45|
|NULL|
|NULL|
|  28|
+----+



`try_parse_json` stands in the same relation to `parse_json`: it returns null on a document that
is not valid JSON, where `parse_json` raises.

In [12]:
broken = spark.createDataFrame(
    [('{"ok": 1}',), ('{"not valid json',)], ["json_text"])

broken.select(
    F.col("json_text"),
    F.try_parse_json("json_text").alias("parsed"),      # null on the malformed line
).show(truncate=False)

+----------------+--------+
|json_text       |parsed  |
+----------------+--------+
|{"ok": 1}       |{"ok":1}|
|{"not valid json|NULL    |
+----------------+--------+



### Discovering the shape of an unfamiliar feed

`schema_of_variant` reports the structure Spark actually found in one value, and
`schema_of_variant_agg` reports the structure across a whole column. On an undocumented feed
this is usually the quickest way to learn what is in it — including the fields nobody mentioned.

In [13]:
docs.select(F.variant_get("v", "$.event_id", "INT").alias("event_id"),
            F.schema_of_variant("v").alias("shape")) \
    .orderBy("event_id").show(truncate=False)

+--------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|event_id|shape                                                                                                                                                                                                                             |
+--------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1       |OBJECT<event_id: BIGINT, items: ARRAY<OBJECT<price: DECIMAL(4,2), qty: BIGINT, sku: STRING>>, ts: STRING, user: OBJECT<age: BIGINT, city: STRING, id: BIGINT>>                                                                    |
|2       |OBJECT<event_id: BIGINT, experiment: O

In [14]:
print("the shape of the whole column, merged:")
docs.select(F.schema_of_variant_agg("v").alias("merged_shape")).show(truncate=False)

the shape of the whole column, merged:
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|merged_shape                                                                                                                                                                                                                                                                            |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|OBJECT<device: OBJECT<os: STRING, push_enabled: BOOLEAN, version: STRING>, event_id: BIGINT, experiment: OBJECT

Read the per-row shapes against each other and the irregularity of the feed is laid out
explicitly: event 2 carries `experiment`, event 5 carries `device`, event 3 has a `user` with no
`age`, and event 4 has a `user.age` of type `STRING` where the others have `BIGINT`. The merged
shape is the union — which is exactly what schema inference computed in section 1, except that
here it is a *value you can inspect* rather than a decision already baked into your table.

---

## 4. The trade-off (Exercise 10)

Both approaches read the same file and answered the same questions. What separates them is
*when* the schema is fixed, and everything else follows from that.

In [15]:
comparison = spark.createDataFrame([
    ("when the schema is fixed",  "before the data is written", "when the data is read"),
    ("an unexpected field",       "silently dropped",           "readable, no migration needed"),
    ("a malformed value",         "null, caught at read time",  "null via try_, or an error"),
    ("passes over the file",      "one",                        "one"),
    ("cost of a new field",       "a schema change everywhere", "none"),
    ("what you give up",          "flexibility as the feed changes", "the early guarantee that the shape is right"),
    ("reaching a nested field",   "dotted path, typed column",  "variant_get with a path and a type"),
], ["question", "declared nested schema", "VARIANT"])

comparison.show(truncate=False)

+------------------------+-------------------------------+-------------------------------------------+
|question                |declared nested schema         |VARIANT                                    |
+------------------------+-------------------------------+-------------------------------------------+
|when the schema is fixed|before the data is written     |when the data is read                      |
|an unexpected field     |silently dropped               |readable, no migration needed              |
|a malformed value       |null, caught at read time      |null via try_, or an error                 |
|passes over the file    |one                            |one                                        |
|cost of a new field     |a schema change everywhere     |none                                       |
|what you give up        |flexibility as the feed changes|the early guarantee that the shape is right|
|reaching a nested field |dotted path, typed column      |variant_get wit

For the feed in Exercise 10 — *most records share a common core while a minority carry
additional, unpredictable ones* — neither column of that table is the whole answer, and the
honest recommendation is to use both at once. Declare the core (`event_id`, `ts`, `user`,
`items`) so that the fields the business depends on are validated at read time and carry real
types, and keep one `VARIANT` column holding the whole document for everything else. The
guarantee is preserved exactly where it earns its keep, and the flexibility exactly where the
feed is genuinely unpredictable.

In [16]:
hybrid = (spark.read.text(EVENTS_PATH).withColumnRenamed("value", "json_text")
          # the agreed core, typed and validated
          .withColumn("core", F.from_json("json_text", event_schema))
          # everything else, kept whole and queryable
          .withColumn("raw",  F.parse_json("json_text"))
          .select(F.col("core.event_id").alias("event_id"),
                  F.col("core.user.city").alias("city"),
                  F.col("core.user.age").alias("age"),
                  F.size("core.items").alias("n_items"),
                  F.try_variant_get("raw", "$.experiment.bucket", "STRING").alias("bucket"),
                  F.try_variant_get("raw", "$.device.os", "STRING").alias("os"),
                  F.col("raw")))

hybrid.drop("raw").orderBy("event_id").show()
hybrid.printSchema()

+--------+-------+----+-------+------+----+
|event_id|   city| age|n_items|bucket|  os|
+--------+-------+----+-------+------+----+
|       1| Boston|  31|      2|  NULL|NULL|
|       2| Lisbon|  45|      1|     B|NULL|
|       3| Boston|NULL|      0|  NULL|NULL|
|       4|Nairobi|NULL|      1|  NULL|NULL|
|       5| Lisbon|  28|      1|  NULL| iOS|
+--------+-------+----+-------+------+----+

root
 |-- event_id: long (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- n_items: integer (nullable = true)
 |-- bucket: string (nullable = true)
 |-- os: string (nullable = true)
 |-- raw: variant (nullable = true)



## Conclusion

Section 3.11 places the two approaches at opposite ends of a spectrum, and the notebook has
walked the same feed down both.

* **Inference** is the option that looks free and is not. It reads the file twice, it makes the
  schema of your table a function of which records were in the sample, and one string in a
  numeric field degrades the whole column to text — silently, and downstream of where you can
  see it.
* **A declared nested schema** is schema-on-write: one pass, real types, an unparseable value
  caught as a null at the moment of reading, and fields outside the contract simply not present.
  What you give up is the ability to absorb a change in the feed without changing the schema.
* **`VARIANT`** is schema-on-read: the whole document kept in a binary encoding that can be
  navigated without re-parsing, any field readable by path whether or not anyone anticipated it,
  and `schema_of_variant` available to discover what actually arrived. What you give up is the
  early guarantee — nothing is validated until something asks for it.

The recurring bargain of the course, in other words, and the right answer depends on whether the
schema is best understood as a stable contract to be enforced or as a moving target to be
tolerated. The hybrid above is what that usually looks like in production: a validated core,
plus one `VARIANT` column for the part of the world that has not agreed on a schema yet.

Two API pairs worth memorising: `parse_json` / `try_parse_json`, and `variant_get` /
`try_variant_get`. In each pair the first raises and the second returns null, and choosing
between them is choosing whether a malformed record should stop the job.